# Move Operator Comparison

This notebook compares `move_first`, `move_best`, and `move_plateau`.

It produces two LaTeX tables:

1. solution quality and winner rate
2. runtime, number of moves, and number of passes

The analysis is grouped by graph type, size class, and density regime.

In [60]:
from pathlib import Path

import numpy as np
import pandas as pd

## Configuration

In [61]:
MOVE_OPERATORS = [
    "move_first",
    "move_best",
    "move_plateau",
]

GRAPH_ORDER = ["powerlaw", "er"]

DATASET_ORDER = [
    "small sparse",
    "small dense",
    "large sparse",
    "large dense",
]

RESULTS_DIR = Path("../results/experiment2/move_operator")

RAW_RESULTS_FILE = RESULTS_DIR / "raw_results.csv"

## Load and verify experiment data

Before the analysis, the notebook lists the contained pipelines, start partitions, zero-gain factors, and number of runs. This provides a compact check that the intended experiment results were loaded.

In [62]:
raw_all = pd.read_csv(RAW_RESULTS_FILE)

required_columns = {
    "pipeline",
    "start_partition",
    "zero_gain_factor",
    "run",
    "graph_type",
    "size_class",
    "regime",
    "dataset",
    "instance",
    "final_density",
    "ls_runtime",
    "num_moves",
    "num_passes",
}

missing_columns = required_columns.difference(raw_all.columns)

if missing_columns:
    raise ValueError("Missing required columns: " + ", ".join(sorted(missing_columns)))

experiment_check = (
    raw_all
    .groupby(
        [
            "pipeline",
            "start_partition",
            "zero_gain_factor",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        num_runs=("run", "nunique"),
        num_instances=("instance", "nunique"),
        num_rows=("run", "size"),
    )
    .sort_values(
        [
            "pipeline",
            "start_partition",
            "zero_gain_factor",
        ],
        na_position="first",
    )
    .reset_index(drop=True)
)

print(f"Loaded {len(raw_all):,} rows from {RAW_RESULTS_FILE}")

experiment_check

Loaded 480,000 rows from ../results/experiment2_new/move_operator/raw_results.csv


,pipeline,start_partition,zero_gain_factor,num_runs,num_instances,num_rows
0,move_best,high_degree_first_matching,NaN,10,2000,20000
1,move_best,high_degree_product_matching,NaN,10,2000,20000
2,move_best,kapoce,NaN,10,2000,20000
3,move_best,leiden_mdgp,NaN,10,2000,20000
4,move_best,matching,NaN,10,2000,20000
5,move_best,maximum_matching,NaN,10,2000,20000
6,move_best,maximum_matching_edge_cover,NaN,10,2000,20000
7,move_best,singleton,NaN,10,2000,20000
8,move_first,high_degree_first_matching,NaN,10,2000,20000
9,move_first,high_degree_product_matching,NaN,10,2000,20000


## Prepare move-operator results

In [63]:
raw_all["dataset_group"] = (
        raw_all["size_class"].astype(str)
        + " "
        + raw_all["regime"].astype(str)
)

raw_all["graph_type"] = pd.Categorical(
    raw_all["graph_type"],
    categories=GRAPH_ORDER,
    ordered=True,
)

raw_all["dataset_group"] = pd.Categorical(
    raw_all["dataset_group"],
    categories=DATASET_ORDER,
    ordered=True,
)

move_raw = raw_all[
    raw_all["pipeline"].isin(MOVE_OPERATORS)
].copy()

present_operators = set(move_raw["pipeline"].dropna().astype(str).unique())
missing_operators = set(MOVE_OPERATORS).difference(present_operators)

if missing_operators:
    raise ValueError("Missing move-operator results: " + ", ".join(sorted(missing_operators)))

move_raw["pipeline"] = pd.Categorical(
    move_raw["pipeline"],
    categories=MOVE_OPERATORS,
    ordered=True,
)

# Solution quality

For every instance and operator, only the run with the highest final solution quality is retained.

The best result across all compared operators is used as the reference. Relative solution quality of an operator is defined as

$
\frac{\text{best solution quality on the instance}}
     {\text{solution quality of the operator}}.
$

A value of $1.0$ means that the operator matches the best result found on the same instance. Values greater than $1.0$ indicate the remaining quality gap.

In [64]:
best_run_keys = [
    "graph_type",
    "dataset_group",
    "dataset",
    "instance",
    "pipeline",
]

best_runs = (
    move_raw
    .sort_values("final_density", ascending=False)
    .groupby(
        best_run_keys,
        observed=True,
        as_index=False,
    )
    .head(1)
    .reset_index(drop=True)
)

In [65]:
instance_keys = [
    "graph_type",
    "dataset_group",
    "dataset",
    "instance",
]

density_table = best_runs.pivot_table(
    index=instance_keys,
    columns="pipeline",
    values="final_density",
    observed=True,
)

density_table = density_table[MOVE_OPERATORS]
density_table.columns.name = None

if density_table.isna().any().any():
    incomplete_instances = density_table[density_table.isna().any(axis=1)]
    raise ValueError(f"{len(incomplete_instances)} instances do not contain results for all move operators.")

In [66]:
best_per_instance = density_table.max(axis=1)

relative_to_best = density_table.rdiv(best_per_instance, axis=0)
relative_to_best.columns.name = "operator"

relative_to_best_summary = (
    relative_to_best
    .groupby(
        level=["graph_type", "dataset_group"]
    )
    .agg(["mean", "min", "max"])
    .stack(level=0, future_stack=True)
    .reset_index()
    .rename(
        columns={
            "mean": "mean_relative_to_best",
            "min": "min_relative_to_best",
            "max": "max_relative_to_best",
        }
    )
)

relative_to_best_summary

,graph_type,dataset_group,operator,mean_relative_to_best,min_relative_to_best,max_relative_to_best
0,powerlaw,small sparse,move_first,1.016997,1.000000,1.055202
1,powerlaw,small sparse,move_best,1.017767,1.000000,1.055202
2,powerlaw,small sparse,move_plateau,1.000000,1.000000,1.000000
3,powerlaw,small dense,move_first,1.009506,1.000000,1.042881
4,powerlaw,small dense,move_best,1.010991,1.000000,1.052486
5,powerlaw,small dense,move_plateau,1.000012,1.000000,1.002918
6,powerlaw,large sparse,move_first,1.011670,1.004678,1.024628
7,powerlaw,large sparse,move_best,1.011699,1.004322,1.024628
8,powerlaw,large sparse,move_plateau,1.000000,1.000000,1.000000
9,powerlaw,large dense,move_first,1.003999,1.000204,1.009468


## Winner rates

An operator is counted as a winner whenever it reaches the best value on an instance. Ties are counted for every involved operator.

In [67]:
is_best = density_table.eq(best_per_instance, axis=0)

best_counts = (
    is_best
    .groupby(
        level=["graph_type", "dataset_group"]
    )
    .sum()
)

best_counts.columns.name = "operator"

num_instances = (
    density_table
    .groupby(level=["graph_type", "dataset_group"])
    .size()
    .rename("num_instances")
    .reset_index()
)

winner_counts = (
    best_counts
    .stack()
    .rename("best_count_with_ties")
    .reset_index()
    .merge(
        num_instances,
        on=["graph_type", "dataset_group"],
    )
)

winner_counts["winner_rate"] = (
        winner_counts["best_count_with_ties"]
        / winner_counts["num_instances"]
)

winner_counts

,graph_type,dataset_group,operator,best_count_with_ties,num_instances,winner_rate
0,powerlaw,small sparse,move_first,1,250,0.004
1,powerlaw,small sparse,move_best,1,250,0.004
2,powerlaw,small sparse,move_plateau,250,250,1.000
3,powerlaw,small dense,move_first,7,250,0.028
4,powerlaw,small dense,move_best,8,250,0.032
5,powerlaw,small dense,move_plateau,249,250,0.996
6,powerlaw,large sparse,move_first,0,250,0.000
7,powerlaw,large sparse,move_best,0,250,0.000
8,powerlaw,large sparse,move_plateau,250,250,1.000
9,powerlaw,large dense,move_first,0,250,0.000


## Runtime

The reported runtime is the total local-search runtime of all runs for one instance and operator, averaged over all instances in the corresponding dataset group.

In [73]:
runtime_per_instance = (
    move_raw
    .groupby(
        best_run_keys,
        observed=True,
    )
    .agg(
        total_runtime=("ls_runtime", "sum"),
        moves_per_run=("num_moves", "mean"),
        passes_per_run=("num_passes", "mean"),
    )
    .reset_index()
)

runtime_summary = (
    runtime_per_instance
    .groupby(
        [
            "graph_type",
            "dataset_group",
            "pipeline",
        ],
        observed=True,
    )
    .agg(
        mean_runtime=("total_runtime", "mean"),
        mean_moves_per_run=("moves_per_run", "mean"),
        mean_passes_per_run=("passes_per_run", "mean"),
    )
    .reset_index()
    .rename(columns={"pipeline": "operator"})
)

runtime_summary

,graph_type,dataset_group,operator,mean_runtime,mean_moves_per_run,mean_passes_per_run
0,powerlaw,small sparse,move_first,3.613431,32.91950,33.91950
1,powerlaw,small sparse,move_best,19.042124,30.16770,31.16770
2,powerlaw,small sparse,move_plateau,12.792214,349.54160,25.40590
3,powerlaw,small dense,move_first,5.554444,28.51580,29.51580
4,powerlaw,small dense,move_best,36.807819,26.11395,27.11395
5,powerlaw,small dense,move_plateau,18.937005,351.66415,16.60415
6,powerlaw,large sparse,move_first,41.693901,211.11080,212.11080
7,powerlaw,large sparse,move_best,996.811346,197.00375,198.00375
8,powerlaw,large sparse,move_plateau,103.475413,2328.52955,21.29505
9,powerlaw,large dense,move_first,303.365740,135.14725,136.14725


## Combined comparison data

In [75]:
comparison_summary = (
    relative_to_best_summary
    .merge(
        winner_counts[
            [
                "graph_type",
                "dataset_group",
                "operator",
                "winner_rate",
            ]
        ],
        on=[
            "graph_type",
            "dataset_group",
            "operator",
        ],
    )
    .merge(
        runtime_summary,
        on=[
            "graph_type",
            "dataset_group",
            "operator",
        ],
    )
)

comparison_summary["operator"] = pd.Categorical(
    comparison_summary["operator"],
    categories=MOVE_OPERATORS,
    ordered=True,
)

comparison_summary = (
    comparison_summary
    .sort_values(
        [
            "graph_type",
            "dataset_group",
            "operator",
        ]
    )
    .reset_index(drop=True)
)

comparison_summary

,graph_type,dataset_group,operator,mean_relative_to_best,min_relative_to_best,max_relative_to_best,winner_rate,mean_runtime,mean_moves_per_run,mean_passes_per_run
0,powerlaw,small sparse,move_first,1.016997,1.000000,1.055202,0.004,3.613431,32.91950,33.91950
1,powerlaw,small sparse,move_best,1.017767,1.000000,1.055202,0.004,19.042124,30.16770,31.16770
2,powerlaw,small sparse,move_plateau,1.000000,1.000000,1.000000,1.000,12.792214,349.54160,25.40590
3,powerlaw,small dense,move_first,1.009506,1.000000,1.042881,0.028,5.554444,28.51580,29.51580
4,powerlaw,small dense,move_best,1.010991,1.000000,1.052486,0.032,36.807819,26.11395,27.11395
5,powerlaw,small dense,move_plateau,1.000012,1.000000,1.002918,0.996,18.937005,351.66415,16.60415
6,powerlaw,large sparse,move_first,1.011670,1.004678,1.024628,0.000,41.693901,211.11080,212.11080
7,powerlaw,large sparse,move_best,1.011699,1.004322,1.024628,0.000,996.811346,197.00375,198.00375
8,powerlaw,large sparse,move_plateau,1.000000,1.000000,1.000000,1.000,103.475413,2328.52955,21.29505
9,powerlaw,large dense,move_first,1.003999,1.000204,1.009468,0.000,303.365740,135.14725,136.14725


## LaTeX helper functions

In [ ]:
def truncate_number(value: float, decimals: int) -> float:
    factor = 10 ** decimals
    return np.trunc(value * factor) / factor


def latex_operator(operator: str) -> str:
    return r"\texttt{" + operator.replace("_", r"\_") + "}"


def format_number(value: float, decimals: int) -> str:
    return f"{value:.{decimals}f}"


def format_percent(value: float, decimals: int = 1) -> str:
    return (
        f"{truncate_number(100 * value, decimals):.{decimals}f}"
        r",\%"
    )

## Build quality LaTeX table

In [80]:
def make_quality_latex_table(df: pd.DataFrame, caption: str, label: str) -> str:
    graph_labels = {
        "powerlaw": "Powerlaw",
        "er": r"Erdős-Rényi",
    }

    lines = [
        r"\begin{table}[t]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\begin{tabular}{lllrr}",
        r"\toprule",
        (
            r"Graphentyp & Datensatz & Operator "
            r"& \shortstack{Mittlere relative\\Lösungsqualität} & Gewinnrate \\"
        ),
        r"\midrule",
    ]

    for graph_type in GRAPH_ORDER:
        graph_df = df[df["graph_type"] == graph_type]

        if graph_df.empty:
            continue

        for dataset_index, dataset in enumerate(DATASET_ORDER):
            part = graph_df[graph_df["dataset_group"] == dataset].copy()

            if part.empty:
                continue

            part["operator"] = pd.Categorical(
                part["operator"],
                categories=MOVE_OPERATORS,
                ordered=True,
            )
            part = part.sort_values("operator")

            for i, row in enumerate(part.itertuples(index=False)):
                graph_cell = (
                    rf"\multirow{{12}}{{*}}"
                    rf"{{{graph_labels[graph_type]}}}"
                    if dataset_index == 0 and i == 0
                    else ""
                )
                dataset_cell = (
                    rf"\multirow{{3}}{{*}}{{{dataset}}}"
                    if i == 0
                    else ""
                )

                mean = format_number(
                    row.mean_relative_to_best,
                    6,
                )
                winner_rate = format_percent(
                    row.winner_rate,
                    1,
                )

                if row.operator == "move_plateau":
                    mean = rf"\textbf{{{mean}}}"
                    winner_rate = rf"\textbf{{{winner_rate}}}"

                lines.append(
                    f"{graph_cell} & {dataset_cell} "
                    f"& {latex_operator(str(row.operator))} "
                    f"& {mean} & {winner_rate} \\\\"
                )

            if dataset_index < len(DATASET_ORDER) - 1:
                lines.append(r"\cmidrule(l){2-5}")
            else:
                lines.append(r"\midrule")

    lines[-1] = r"\bottomrule"
    lines.extend(
        [
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)


quality_latex = make_quality_latex_table(
    comparison_summary,
    caption=(
        "Mittlere relative Lösungsqualität und Gewinnrate der Move-Operatoren auf Powerlaw- und Erdős-Rényi-Instanzen. Die relative Lösungsqualität ist der Quotient aus der besten auf derselben Instanz gefundenen Lösung und der Lösung des jeweiligen Operators. Ein Wert von 1 entspricht der besten gefundenen Lösung."
    ),
    label="tab:move_quality",
)

print(quality_latex)

\begin{table}[t]
\centering
\caption{Mittlere relative Lösungsqualität und Gewinnrate der Move-Operatoren auf Powerlaw- und Erdős-Rényi-Instanzen. Die relative Lösungsqualität ist der Quotient aus der besten auf derselben Instanz gefundenen Lösung und der Lösung des jeweiligen Operators. Ein Wert von 1 entspricht der besten gefundenen Lösung.}
\label{tab:move_quality}
\begin{tabular}{lllrr}
\toprule
Graphentyp & Datensatz & Operator & \shortstack{Mittlere relative\\Lösungsqualität} & Gewinnrate \\
\midrule
\multirow{12}{*}{Powerlaw} & \multirow{3}{*}{small sparse} & \texttt{move\_first} & 1.016997 & 0.4,\% \\
 &  & \texttt{move\_best} & 1.017767 & 0.4,\% \\
 &  & \texttt{move\_plateau} & \textbf{1.000000} & \textbf{100.0,\%} \\
\cmidrule(l){2-5}
 & \multirow{3}{*}{small dense} & \texttt{move\_first} & 1.009506 & 2.8,\% \\
 &  & \texttt{move\_best} & 1.010991 & 3.2,\% \\
 &  & \texttt{move\_plateau} & \textbf{1.000012} & \textbf{99.6,\%} \\
\cmidrule(l){2-5}
 & \multirow{3}{*}{large spa

## Build runtime LaTeX table

In [ ]:
def make_runtime_latex_table(df: pd.DataFrame, caption: str, label: str) -> str:
    graph_labels = {
        "powerlaw": "Powerlaw",
        "er": r"Erdős-Rényi",
    }

    lines = [
        r"\begin{table}[t]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\begin{tabular}{lllrrr}",
        r"\toprule",
        (
            r"Graphentyp & Datensatz & Operator & Laufzeit (s) "
            r"& Moves & Durchläufe \\"
        ),
        r"\midrule",
    ]

    for graph_type in GRAPH_ORDER:
        graph_df = df[df["graph_type"] == graph_type]

        for dataset_index, dataset in enumerate(DATASET_ORDER):
            part = graph_df[graph_df["dataset_group"] == dataset].copy()

            part["operator"] = pd.Categorical(
                part["operator"],
                categories=MOVE_OPERATORS,
                ordered=True,
            )
            part = part.sort_values("operator")

            if part.empty:
                continue

            for i, row in enumerate(part.itertuples(index=False)):
                graph_cell = (
                    rf"\multirow{{12}}{{*}}"
                    rf"{{{graph_labels[graph_type]}}}"
                    if dataset_index == 0 and i == 0
                    else ""
                )
                dataset_cell = (
                    rf"\multirow{{3}}{{*}}{{{dataset}}}"
                    if i == 0
                    else ""
                )

                lines.append(
                    f"{graph_cell} & {dataset_cell} "
                    f"& {latex_operator(str(row.operator))} "
                    f"& {format_number(row.mean_runtime, 2)} "
                    f"& {format_number(row.mean_moves_per_run, 1)} "
                    f"& {format_number(row.mean_passes_per_run, 1)} \\\\"
                )

            if dataset_index < len(DATASET_ORDER) - 1:
                lines.append(r"\cmidrule(l){2-6}")
            else:
                lines.append(r"\midrule")

    lines[-1] = r"\bottomrule"
    lines.extend(
        [
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)


runtime_latex = make_runtime_latex_table(
    comparison_summary,
    caption=(
        "Mittlere Gesamtlaufzeit der zehn Runs sowie durchschnittliche Anzahl ausgeführter Knotenverschiebungen und Durchläufe auf Powerlaw- und Erdős-Rényi-Instanzen."
    ),
    label="tab:move_runtime",
)

print(runtime_latex)